# Human-in-the-Loop (HITL) — SQL 에이전트로 배우기

에이전트가 도구를 **실행하기 직전에 멈추고, 사람이 결정한 뒤 이어서 실행**하는 Human-in-the-Loop를 배웁니다.
예제로는 Chinook 데이터베이스에 SQL을 실행하는 에이전트를 사용합니다.

**참고**:
- [LangChain 공식 문서 - Human-in-the-Loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangChain 공식 문서 - SQL Agent](https://docs.langchain.com/oss/python/langchain/sql-agent)

## 학습 내용
- `HumanInTheLoopMiddleware`로 특정 도구 실행 전에 일시정지(interrupt)하기
- 사람의 결정 3가지: **approve**(승인) / **edit**(수정 후 실행) / **reject**(거부)
- `allowed_decisions`로 도구별 허용 결정 제한
- 체크포인터와 `thread_id`가 필요한 이유

> SQL 도구 설계(쿼리 검증, 리소스 등)는 **105_DB_MCP_Agent**에서, Deep Agent의 HITL(`interrupt_on`)은 **401_Deep_Agent_Advanced**에서 다룹니다.

## 1. 예제 준비 — SQL 도구 3개

HITL을 보여주기 위한 최소한의 SQL 도구를 `@tool`로 만듭니다. 이 노트북의 주제는 HITL이므로 도구는 단순하게 둡니다.

| 도구 | 역할 | 승인 필요 |
|------|------|------|
| `sql_db_list_tables` | 테이블 목록 조회 | ✕ |
| `sql_db_schema` | 테이블 스키마 조회 | ✕ |
| `sql_db_query` | SQL 실행 | **○** |

데이터베이스는 **읽기 전용**으로 연결합니다. 사람이 실수로 위험한 쿼리를 승인하더라도 데이터는 바뀌지 않습니다.

Chinook 데이터베이스가 없으면 다음 명령어로 만들 수 있습니다:
```bash
curl -s https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql | sqlite3 Chinook.db
```

In [ ]:
def connect_readonly() -> sqlite3.Connection:
def sql_db_list_tables() -> str:
def sql_db_schema(table_names: str) -> str:
def sql_db_query(query: str) -> str:

## 2. Human-in-the-Loop 동작 원리

```
사용자 질문 → 모델이 도구 호출 결정 → [HITL 미들웨어가 실행 직전 가로챔] → ⏸ interrupt (상태 저장)
                                                                          ↓
                       도구 실행 / 수정 실행 / 거부 메시지 전달  ←  사람의 결정: Command(resume=...)
```

| 결정 | 의미 | resume 값 |
|------|------|------|
| **approve** | 모델이 만든 호출을 그대로 실행 | `{"type": "approve"}` |
| **edit** | 사람이 인자를 고쳐서 실행 | `{"type": "edit", "edited_action": {"name": ..., "args": {...}}}` |
| **reject** | 실행하지 않고 이유를 모델에게 전달 → 모델이 다시 시도 | `{"type": "reject", "message": "..."}` |
| respond | 실행하지 않고 사람이 결과를 대신 답함 ("사용자에게 묻기" 같은 도구용) | `{"type": "respond", "message": "..."}` |

- **체크포인터가 필수**입니다: 멈춘 시점의 대화 상태를 저장해 두었다가, 결정을 받으면 그 지점부터 이어서 실행합니다.
- **같은 `thread_id`로 재개**해야 합니다: thread_id가 "어느 멈춘 대화를 이어갈지"를 가리킵니다.

## 3. HITL 에이전트 만들기

`interrupt_on`에 도구별 설정을 지정합니다.
- `False`: 승인 없이 실행
- `True`: 승인 필요 (모든 결정 허용)
- `{"allowed_decisions": [...]}`: 승인 필요 + 허용할 결정만 지정

실행 결과를 보기 쉽게 출력하는 도우미 함수입니다. 멈췄으면 **대기 중인 작업과 허용된 결정**을, 끝났으면 **최종 답변**을 보여줍니다.

In [ ]:
def run(agent, payload, config):

## 4. 사람의 결정 3가지

### 4.1 approve — 그대로 실행

에이전트는 테이블·스키마 조회는 바로 실행하고, `sql_db_query` 직전에 멈춥니다.

쿼리를 확인했으면 `approve`로 재개합니다. **같은 config(thread_id)**를 넘겨야 멈춘 대화가 이어집니다.

### 4.2 edit — 쿼리를 고쳐서 실행

모델이 만든 쿼리 대신 **사람이 고친 쿼리**를 실행합니다. 예를 들어 결과 개수를 줄이거나 조건을 바꿀 때 사용합니다.
모델에게는 실행 결과와 함께 "사람이 호출을 바꿔 실행했다"는 안내가 전달되어, 원래 쿼리를 다시 실행하지 않습니다.

In [ ]:
# 사람이 직접 고친 쿼리: 3개만, 같은 가격이면 TrackId 순으로

### 4.3 reject — 거부하고 이유 알려주기

거부하면 도구는 실행되지 않고, **거부 사유가 모델에게 전달**됩니다. 모델은 사유를 반영해 새 쿼리를 만들고, 그 쿼리도 다시 승인을 기다립니다.
아래에서는 개인정보(이메일)를 조회하려는 쿼리를 거부합니다.

모델이 거부 사유를 반영한 **새 쿼리**로 다시 승인을 요청합니다. 확인 후 승인합니다.

## 5. 정리

| 구성 요소 | 역할 |
|------|------|
| `HumanInTheLoopMiddleware(interrupt_on=...)` | 지정한 도구의 실행 직전에 일시정지 |
| `allowed_decisions` | 도구별로 허용할 결정(approve / edit / reject / respond) 제한 |
| 체크포인터 + `thread_id` | 멈춘 상태를 저장하고, 같은 thread_id로 이어서 실행 |
| `Command(resume={"decisions": [...]})` | 사람의 결정을 전달해 재개 (대기 중인 호출 순서대로) |

- **HITL은 마지막 방어선이 아닙니다**: 사람도 실수할 수 있으므로 읽기 전용 연결 같은 권한 제한과 함께 사용합니다.
- 다음 단계: **105_DB_MCP_Agent** — SQL 도구를 MCP 서버로 분리하고 쿼리 검증까지 갖추기 / **401_Deep_Agent_Advanced** — Deep Agent에서 `interrupt_on` 파라미터로 HITL 사용하기

### 실습 문제

본문에서 배운 HITL을 활용해 다음 두 가지를 완성하세요. (새 `thread_id`를 사용하세요)

1. **edit로 결과 개수 바꾸기**:
   `"판매 수량이 가장 많은 트랙을 알려줘"`를 실행해 `sql_db_query` 승인 대기에서 멈추면,
   대기 중인 쿼리의 `LIMIT`을 **3**으로 바꾼 쿼리로 `edit` 해서 실행하세요.
   (`interrupt.value["action_requests"][0]["args"]["query"]`에서 모델이 만든 쿼리를 꺼낼 수 있습니다)

2. **reject 후 재시도 흐름**:
   `"직원들의 이름, 생년월일, 주소를 보여줘"`를 실행해 멈추면,
   `"생년월일과 주소는 개인정보라 조회할 수 없습니다. 이름과 직책만 보여주세요."`로 **reject** 하고,
   모델이 다시 만든 쿼리를 **approve** 해서 최종 답변까지 출력하세요.

### 테스트 입력 예시

* 1번: 최종 답변에 트랙 **3개**가 나오는지 확인
* 2번: 두 번째 승인 대기의 쿼리에 `BirthDate`, `Address`가 **없고** `Title`(직책)이 있는지 확인